# Multiclass Classification Export Example

This notebook provides an end-to-end example on how to train and export a PyTorch classification model to a format accepted by uniVision Module Image ONNX. This will allow you to deploy your models on the Wenglor's B60 Smart Camera and MVC Machine Vision Controller.

DISCLAIMER: Focus of this notebook is to provide an example for the export and quantization. The training code is provided purely for illustration and not meant to provide a production-grade model. You will want to optimize the training pipeline and add more images. Depending on your situation, you can start from one of the following steps: 
1. Train PyTorch model.
    * If you don't have a trained model, start here. 
2. Export PyTorch model to ONNX.
    * If you already have a trained PyTorch model, start here.
3. Quantize ONNX model (optional)
    * If you have a trained ONNX model and it supports quantization, e.g. the model is in the list of quantization supported models (see below), start here. For this step, you will also need to provide a dataset that your model had been trained on. This dataset will be used for a calibration step during model quantization. 
4. Export ONNX model.
    * If your ONNX model does not support quantization or you cannot provide a calibration dataset, begin here. Similarly, if you plan to deploy your unquantized model on the MVC Machine Vision Controller, this is also the right starting point.  
  
The models below have been tested and are known to run successfully:
* `torchvision.models.quantization.resnet18`
* `torchvision.models.quantization.resnet50`
* `torchvision.models.regnet_x_400mf`
* `torchvision.models.regnet_x_800mf`
* (recommended) `torchvision.models.regnet_x_1_6gf`
* (recommended) `torchvision.models.regnet_x_3_2gf`

We recommend a maximum input image size of 416×416 to stay within the memory and latency constraints of the B60 Smart Camera.

Other models may also be supported either through execution on the CPU or even hardware-accelerated. Please report your findings and we will update this list.

**What is quantization and why do we need it?**

Quantization is a technique used to make machine learning models run faster and more efficiently, especially on smaller devices with limited resources, like smartphones or smart cameras. By reducing the precision of the numbers the model uses, it can significantly decrease the amount of memory and computing power required, without losing too much accuracy. For example, our benchmark results show that a quantized regnet_x_1_6gf model running on a B60 camera achieves an inference speed of 50 milliseconds, which is nearly 7 times faster than the unquantized version (340 milliseconds). 

## 1. Train PyTorch model 

In [ ]:
# https://github.com/pytorch/pytorch/issues/140765
import getpass
import os

# Add dummy username so default_cache_dir() doesn't blow up
getpass.getuser = lambda: "torchuser"

# Also set the cache dir explicitly
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor_cache"

In [ ]:
import glob
import os
import time
import uuid
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch
import torchvision
from onnxruntime.quantization.quantize import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torcheval.metrics import MulticlassAccuracy
from torchvision.models import (
    RegNet_X_1_6GF_Weights,
    RegNet_X_3_2GF_Weights,
    RegNet_X_16GF_Weights,
    RegNet_X_400MF_Weights,
    RegNet_X_800MF_Weights,
    regnet_x_1_6gf,
    regnet_x_3_2gf,
    regnet_x_16gf,
    regnet_x_400mf,
    regnet_x_800mf,
)
from torchvision.transforms import v2 as T
from utils.constants import IMAGENET_MEAN, IMAGENET_STD
from utils.enums import DatasetColorMode
from utils.export import export_univision_model_v3
from utils.heatmap import get_heatmap_feature_layer
from utils.image import is_rgb_image, read_and_resize_input_example, read_image_file
from utils.quantization import (
    TorchCalibrationDataReader,
    get_nodes_to_exclude,
    sort_nodes_topologically,
)

In [ ]:
IMAGE_SIZE = (224, 224)
EPOCHS = 50
BATCH_SIZE = 32
NUM_WORKERS = 4
BASE_LEARNING_RATE = 1e-3

DATA_ROOT = Path("../data")
IMAGE_DIR = DATA_ROOT / "images/multi-class"
MODEL_DIR = DATA_ROOT / "model"

MODEL_DIR.mkdir(exist_ok=True, parents=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

### 1.1 Get dataframe

For this example, we will use a small rock-paper-scissors dataset consisting of 200 images across 3 classes.

In [ ]:
image_paths = sorted(list(IMAGE_DIR.rglob("*.bmp")))
len(image_paths), image_paths[:2]

In [ ]:
classes = sorted(set([image_path.parents[0].name for image_path in image_paths]))
classes

In [ ]:
df = pd.DataFrame({"image_path": image_paths})

df["target"] = df.image_path.apply(lambda x: x.parents[0].name)
df["target"] = df.target.map({v: k for k, v in enumerate(classes)})

print(df.shape)
df.head(2)

In [ ]:
df.target.value_counts()

In [ ]:
train, valid = train_test_split(df, test_size=0.1, random_state=0, stratify=df.target)
train.shape, valid.shape

### 1.2 Create dataset

Define image preprocessing and augmentations. For more information on image preprocessing and normalization, please see Export ONNX model. For this example, we will rescale input values to the `[0, 1]` range. For image augmentations, we use random changes in brightness and contrast, along with image rotating and scaling. Depending on your case, different preprocessing and augmentations could be applied. 

In [ ]:
train_transform = T.Compose(
    [
        T.ToImage(),
        T.Resize(IMAGE_SIZE),
        T.ColorJitter(brightness=0.5, contrast=0.5),
        T.RandomAffine(degrees=(-180, 180), scale=(0.8, 1.2)),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
valid_transform = T.Compose(
    [
        T.ToImage(),
        T.Resize(IMAGE_SIZE),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

In [ ]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        image, target = (
            read_image_file(self.df.image_path.iloc[index]),
            self.df.target.iloc[index],
        )

        image = (
            cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            if len(image.shape) == 2 or image.shape[2] == 1
            else image
        )

        return self.transform(Image.fromarray(image)), target

In [ ]:
train_dataset = ImageDataset(train, train_transform)
valid_dataset = ImageDataset(valid, valid_transform)

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    prefetch_factor=4,
    pin_memory=DEVICE.type == "cuda",
)
valid_dataloader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
    prefetch_factor=4,
    pin_memory=DEVICE.type == "cuda",
)

### 1.3 Plot images

Check that the selected image preprocessing and augmentations work correctly.  

In [ ]:
def plot_images(batch, rows, columns):

    plt.figure(
        figsize=(IMAGE_SIZE[1] // 12, IMAGE_SIZE[0] // 12 * rows // columns),
        frameon=False,
    )

    for index in range(rows * columns):

        plt.subplot(rows, columns, index + 1)
        plt.axis("off")
        plt.imshow(batch[0][index].numpy().transpose(1, 2, 0), aspect="auto")
        plt.text(
            IMAGE_SIZE[1] // 2,
            IMAGE_SIZE[0] - 10,
            classes[batch[1][index].numpy()],
            size=10,
            ha="center",
            color="k",
            bbox=dict(boxstyle="round, pad=0", facecolor="w", edgecolor="w"),
        )
    plt.subplots_adjust(hspace=0, wspace=0)
    plt.show()

In [ ]:
plot_images(next(iter(train_dataloader)), rows=2, columns=6)

In [ ]:
plot_images(next(iter(valid_dataloader)), rows=2, columns=6)

### 1.4 Train model

If you're looking for a model that supports quantization, see the list of quantization supported models at the beginning of the notebook. Otherwise, any model that supports ONNX export with the opset version 11 can be used. 

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, classes, return_logits):
        super(Model, self).__init__()

        self.classes = classes

        # We recommend regnet_x_1_6gf or regnet_x_3_2gf
        # self.backbone = regnet_x_1_6gf(weights=RegNet_X_1_6GF_Weights.IMAGENET1K_V2)
        self.backbone = regnet_x_3_2gf(weights=RegNet_X_3_2GF_Weights.IMAGENET1K_V2)

        self.backbone.fc = torch.nn.Linear(
            self.backbone.fc.in_features, out_features=len(classes)
        )
        self.softmax = torch.nn.Softmax(dim=1)
        self.return_logits = return_logits

    def forward(self, x):
        x = self.backbone(x)
        if self.return_logits:
            return x
        return self.softmax(x)

In [ ]:
def train_model(model, dataloaders):

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=BASE_LEARNING_RATE,
        steps_per_epoch=len(dataloaders["train"]),
        epochs=EPOCHS,
    )
    metric = MulticlassAccuracy(average="macro", num_classes=len(classes))
    valid_acc = 0
    best_acc = 0
    for epoch in range(EPOCHS):

        epoch_log = [f"epoch {epoch+1:04d}/{EPOCHS:04d}"]
        time_start = time.monotonic()
        for phase in ["train", "valid"]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0
            metric.reset()

            for inputs, target in dataloaders[phase]:
                inputs = inputs.to(
                    DEVICE,
                    memory_format=torch.channels_last,
                    non_blocking=(DEVICE.type == "cuda"),
                )
                target = target.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(phase == "train"):
                    output = model(inputs)
                    loss = criterion(output, target)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                        scheduler.step()

                running_loss += loss.item() * inputs.size(0)
                metric.update(torch.argmax(output, 1), target)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            phase_acc = metric.compute().item()

            if phase == "valid":
                valid_acc = phase_acc
                if valid_acc >= best_acc:
                    best_acc = valid_acc
                    best_weights = {k: v.clone() for k, v in model.state_dict().items()}

            epoch_log.append(
                f"{phase}_loss: {epoch_loss:.4f} {phase}_balanced_accuracy: {phase_acc:.4f}"
            )

        epoch_log.append(
            f"lr: {scheduler.get_last_lr()[0]:.6f}, duration: {time.monotonic() - time_start}"
        )
        print(" ".join(epoch_log))

    torch.save(best_weights, MODEL_DIR / "best_weights.pth")
    model.load_state_dict(best_weights)
    print(f"Restored best checkpoint: valid_balanced_accuracy = {best_acc:.4f}")

In [ ]:
dataloaders = {
    "train": train_dataloader,
    "valid": valid_dataloader,
}

model = Model(classes, return_logits=True)
model.to(
    DEVICE, memory_format=torch.channels_last, non_blocking=(DEVICE.type == "cuda")
)

train_model(model, dataloaders)

In [ ]:
# Reinitialize the model with a softmax layer to return probabilities.
model = Model(classes, return_logits=False)
model.to(DEVICE)
model.eval()

model.load_state_dict(torch.load(MODEL_DIR / "best_weights.pth"))

## 2. Export PyTorch model to ONNX

If you already have a trained PyTorch model, start here.

**Note:** Quantizing for **NPU** acceleration may fail if your model contains unsupported layers/operators.  
Only the models listed at the start of this notebook are guaranteed to work in the full NPU workflow.

Alternatively, you can quantize and run on **CPU** instead. This still gives significant speed-ups with much broader layer support. See `inference_device` in [metadata.md](../metadata.md), this can be specifed at the final step of this notebook.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
FP32_ONNX_MODEL_PATH = MODEL_DIR / "fp32_model.onnx"

In [ ]:
torch.onnx.export(
    model,
    torch.randn((1, 3, *IMAGE_SIZE)).to(DEVICE),
    FP32_ONNX_MODEL_PATH,
    export_params=True,
    verbose=False,
    input_names=["input"],
    output_names=["output"],
    opset_version=20,
    do_constant_folding=True,
    dynamo=False,
    external_data=False,
    optimize=True,
)

## 3. Quantize ONNX model (optional)

If you already have a trained ONNX model that supports quantization, start here. This step also requires you to define your training and validation datasets and dataloaders. Please note that if your model has a softmax layer it will be excluded from quantization.

In [ ]:
INT8_ONNX_MODEL_PATH = MODEL_DIR / "int8_model.onnx"

In [ ]:
dir_path = FP32_ONNX_MODEL_PATH.parent
stem = FP32_ONNX_MODEL_PATH.stem
suffix = FP32_ONNX_MODEL_PATH.suffix
preprocessed_fp32_onnx_model_path = dir_path / f"preprocessed_{stem}{suffix}"

quant_pre_process(
    FP32_ONNX_MODEL_PATH,
    preprocessed_fp32_onnx_model_path,
)

calibration_data_reader = TorchCalibrationDataReader(
    preprocessed_fp32_onnx_model_path,
    dataset=train_dataset,
    batch_size=1,
    shuffle=True,
    samples=500,
)
quantize_static(
    preprocessed_fp32_onnx_model_path,
    INT8_ONNX_MODEL_PATH,
    calibration_data_reader=calibration_data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
    calibrate_method=CalibrationMethod.MinMax,
    nodes_to_exclude=get_nodes_to_exclude(
        onnx.load(preprocessed_fp32_onnx_model_path)
    ),  # comment this for faster inference speed at the cost of quality degradation
)

onnx_model = onnx.load(INT8_ONNX_MODEL_PATH)
sort_nodes_topologically(onnx_model)
onnx.save(onnx_model, INT8_ONNX_MODEL_PATH)

print(f"Model quantized and saved to: {INT8_ONNX_MODEL_PATH}")

### 3.1 Compare FP32 and INT8 models (optional)

If you have a validation dataset, you can measure the performance of your model before and after quantization.  

In [ ]:
valid_dataloader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    persistent_workers=True,
)

In [ ]:
providers = ["CPUExecutionProvider"]
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

In [ ]:
phase = {"FP32": FP32_ONNX_MODEL_PATH, "INT8": INT8_ONNX_MODEL_PATH}

for phase_name, model_path in phase.items():
    session = ort.InferenceSession(model_path, sess_options, providers=providers)

    input_name = session.get_inputs()[0].name
    output_names = [output.name for output in session.get_outputs()]

    y_true, y_pred = [], []
    for inputs, target in valid_dataloader:
        inputs = {input_name: inputs.numpy()}
        output = session.run(output_names, inputs)

        y_true.append(target)
        y_pred.append(np.argmax(output))
    y_true, y_pred = np.concatenate(y_true), np.array(y_pred)

    print(
        f"{phase_name} Balanced accuracy score: {balanced_accuracy_score(y_true, y_pred):.02%}"
    )

## 4. Export ONNX model

### 4.1 YAML format

During model export, a metadata .yaml file is generated, it contains all the necessary information about the model such as the input size, class names, etc. It is essential to use the right parameters to ensure that correctness of the preprocessing. Please review the example metadata file and its description below. For more details, refer to [metadata.md](../metadata.md)

#### 4.1.1 `.yaml` metadata file example

#### 4.1.2 Input preprocessing order

1. (optional) Color space/channel transformation

    As a result of this transformation the image data has shape
    
        a. (batch_size, original_height, original_width, channels) for channel_order=NHWC
      
        b. (batch_size, channels, original_height, original_width) for channel_order=NCHW

   where channels is 1 or 3 and the image channel data matches the order defined by color_space (RGB/BGR/GRAYSCALE).

3. Image resize

4. (optional) Unit scaling.

    `x = x / 255`

   
5. (optional) Standardization

   `x = (x - standardization_mean) / standardization_std`

### 4.2 Heatmap feature layer (optional)

Saliency map generation methods help us understand which parts of an image or a machine learning model focuses on to make its decisions. This makes AI models more transparent and easier to trust because we can see what is important to the model when it makes certain predictions.

In [ ]:
# For FP32 ONNX model
heatmap_feature_layer_fp32 = get_heatmap_feature_layer(
    "RegNet_X_3_2GF", quantization=False
)
heatmap_feature_layer_fp32

In [ ]:
# For INT8 ONNX model
heatmap_feature_layer_int8 = get_heatmap_feature_layer(
    "RegNet_X_3_2GF", quantization=True
)
heatmap_feature_layer_int8

### 4.3 Export

We can export either an FP32 or an INT8 ONNX model. The result is the `.onnx` file together with a metadata file, zipped together as a `.u3o` file. In univision, the `.u3o` file can be loaded with the `module image onnx`.
Please see the examples below.

We also need to define a dataset color mode. For more information please refer to the `.yaml` metadata file description. 

In [ ]:
if any(is_rgb_image(path) for path in df.image_path):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
# Define an input example. If you cannot provide an example, an array of zeros can be used.
input_example = read_and_resize_input_example(
    df.image_path.iloc[0], IMAGE_SIZE, dataset_color_mode
)

#### 4.3.1 Export FP32 ONNX model

In [ ]:
UNIVISION_FP32_MODEL_PATH = MODEL_DIR / "example-model-fp32.u3o"

In [ ]:
export_univision_model_v3(
    univision_model_path=UNIVISION_FP32_MODEL_PATH,
    onnx_model_path=FP32_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,  # or e.g. "example_model_name"
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization=None,
    resize_mode="STRETCH",
    unit_scaling=True,
    standardization_std=IMAGENET_STD,
    standardization_mean=IMAGENET_MEAN,
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space="RGB",
    output_type="MULTI_CLASS_CLASSIFICATION",
    heatmap_feature_layer=heatmap_feature_layer_fp32,  # or None
)

#### 4.3.2 Export INT8 ONNX model

In [ ]:
UNIVISION_INT8_MODEL_PATH = MODEL_DIR / "example-model-int8.u3o"

In [ ]:
export_univision_model_v3(
    univision_model_path=UNIVISION_INT8_MODEL_PATH,
    onnx_model_path=INT8_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,  # or e.g. "example_model_name"
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization="INT8",
    resize_mode="STRETCH",
    unit_scaling=True,
    standardization_std=IMAGENET_STD,
    standardization_mean=IMAGENET_MEAN,
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space="RGB",
    output_type="MULTI_CLASS_CLASSIFICATION",
    heatmap_feature_layer=heatmap_feature_layer_int8,  # or None
)